In [1]:
# Standard Library
import os
import sys
import json
import math
import time
import shutil
from pathlib import Path
from collections import Counter

# Third-Party Libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
import kagglehub
from ultralytics import YOLO

# PyTorch Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import joblib



Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# Download latest version
path = kagglehub.dataset_download("shahliza27/ur-fall-detection-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ur-fall-detection-dataset' dataset.
Path to dataset files: /kaggle/input/ur-fall-detection-dataset


In [4]:
#-------------------------------------------------------------------------------
# Copying datasets into content/ from session's cache/
#-------------------------------------------------------------------------------

# Source directories
URFALL_SRC = r"/kaggle/input/ur-fall-detection-dataset/UR_fall_detection_dataset_cam0_rgb"
# Target directories
URFALL_DST = "/content/UR_FALL"

def move_dataset(src, dst):
    if os.path.exists(dst):
        print(f"{dst} already exists. Skipping copy.")
    else:
        shutil.copytree(src, dst)
        print(f"Copied dataset to {dst}")

# Move datasets
move_dataset(URFALL_SRC, URFALL_DST)


Copied dataset to /content/UR_FALL


In [5]:
import os
import shutil
import numpy as np

SOURCE_ROOT = "/content/UR_FALL"
DEST_ROOT   = "/content/UR_FALL_32"

os.makedirs(DEST_ROOT, exist_ok=True)

for folder in sorted(os.listdir(SOURCE_ROOT)):
    src_folder = os.path.join(SOURCE_ROOT, folder)
    if not os.path.isdir(src_folder):
        continue

    dst_folder = os.path.join(DEST_ROOT, folder)
    os.makedirs(dst_folder, exist_ok=True)

    images = sorted([
        f for f in os.listdir(src_folder)
        if f.endswith(".png") or f.endswith(".jpg")
    ])

    N = len(images)
    if N < 32:
        print(f"Skipping {folder}, only {N} frames")
        continue

    indices = np.linspace(0, N - 1, 32, dtype=int)

    for idx in indices:
        shutil.copy(
            os.path.join(src_folder, images[idx]),
            os.path.join(dst_folder, images[idx])
        )

print("✅ Frame extraction completed.")

✅ Frame extraction completed.


In [6]:
STAGE1_DIR = "/content/stage1_output"
STAGE2_DIR = "/content/stage2_output"
STAGE3_DIR = "/content/stage3_output"
STAGE4_DIR = "/content/stage4_output"

for d in [STAGE1_DIR, STAGE2_DIR, STAGE3_DIR, STAGE4_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Stage output directories created.")

✅ Stage output directories created.


In [7]:
# ================================================================
# MODEL DEFINITIONS — Stage1 3DCNN  |  Stage3 FallAttentionModel
# ================================================================

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLIP_LEN   = 32
FRAME_SIZE = 224
SEQ_LEN      = 16
NUM_FEATURES = 38   # 17 keypoints * 2 coords = 34 + 4 physics features

# ---- Stage 1: 3D-CNN ----
class Stage1_3DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv3d(3,64,(3,7,7),(1,2,2),(1,3,3),padding_mode='replicate')
        self.bn1   = nn.BatchNorm3d(64)
        self.pool1 = nn.MaxPool3d((1,3,3),(1,2,2),(0,1,1))
        self.conv2 = nn.Conv3d(64,128,3,padding=1,padding_mode='replicate')
        self.bn2   = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(2,2)
        self.conv3 = nn.Conv3d(128,256,3,padding=1,padding_mode='replicate')
        self.bn3   = nn.BatchNorm3d(256)
        self.conv4 = nn.Conv3d(256,256,3,padding=1,padding_mode='replicate')
        self.bn4   = nn.BatchNorm3d(256)
        self.global_pool = nn.AdaptiveAvgPool3d((1,1,1))
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(256,2)

    def forward(self, x, return_saliency=False):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))

        sal = x.detach().pow(2).mean(dim=(1,3,4))
        T_prime = sal.shape[1]
        edge_mask = torch.ones(T_prime).to(sal.device)
        edge_mask[0] = 0.05
        edge_mask[-1] = 0.5
        sal = sal * edge_mask
        min_val = sal.min(dim=1,keepdim=True)[0]
        max_val = sal.max(dim=1,keepdim=True)[0]
        sal = (sal - min_val) / (max_val - min_val + 1e-6)
        sal = F.softmax(sal * 5.0, dim=1)

        x = F.relu(self.bn4(self.conv4(x)))
        x = self.global_pool(x).flatten(1)
        x = self.dropout(x)
        logits = self.fc(x)
        if return_saliency:
            return logits, sal
        return logits

# ---- Stage 3: Attention Model ----
class FallAttentionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim=NUM_FEATURES, num_heads=2, batch_first=True)
        self.norm = nn.LayerNorm(NUM_FEATURES)
        self.fc = nn.Sequential(
            nn.Linear(NUM_FEATURES,64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64,1), nn.Sigmoid()
        )

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attention(x, x, x, key_padding_mask=mask)
        x = self.norm(x + attn_out)
        if mask is not None:
            wm = (~mask).unsqueeze(-1).float()
            x_mean = (x * wm).sum(dim=1) / (wm.sum(dim=1) + 1e-8)
        else:
            x_mean = torch.mean(x, dim=1)
        out = self.fc(x_mean)
        return out, attn_weights

print('✅ Model classes defined. Device:', DEVICE)


✅ Model classes defined. Device: cuda


In [8]:
# ================================================================
# LOAD ALL PRE-TRAINED MODELS (run once before the pipeline loop)
# ================================================================

# ---------- Paths — update if filenames differ ----------
STAGE1_CKPT = '/content/stage1.pth'       # Stage-1 3D-CNN
STAGE3_CKPT = '/content/stage3.pth'      # Stage-3 Attention model

# ---- Stage 1 ----
stage1_model = Stage1_3DCNN().to(DEVICE)
stage1_model.load_state_dict(torch.load(STAGE1_CKPT, map_location=DEVICE))
stage1_model.eval()
print('✅ Stage-1 3D-CNN loaded')

# ---- Stage 3 ----
stage3_model = FallAttentionModel().to(DEVICE)
stage3_model.load_state_dict(torch.load(STAGE3_CKPT, map_location=DEVICE))
stage3_model.eval()
print('✅ Stage-3 Attention model loaded')

# ---- MiDaS (Stage-4 depth) ----
print('Loading MiDaS...')
midas = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small')
midas.to(DEVICE)
midas.eval()
midas_transforms = torch.hub.load('intel-isl/MiDaS', 'transforms')
midas_transform  = midas_transforms.small_transform
print('✅ MiDaS loaded')

# ---- YOLO (Stage-2 pose) ----
yolo_model = YOLO('yolov8m-pose.pt')
print('✅ YOLOv8 pose model loaded')



✅ Stage-1 3D-CNN loaded
✅ Stage-3 Attention model loaded
Loading MiDaS...


/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip
Loading weights:  None
Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to /root/.cache/torch/hub/checkpoints/midas_v21_small_256.pt


100%|██████████| 81.8M/81.8M [00:02<00:00, 28.8MB/s]
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


✅ MiDaS loaded
✅ YOLOv8 pose model loaded


In [11]:
# ================================================================
# PIPELINE STAGE FUNCTIONS
# ================================================================
from scipy.ndimage import gaussian_filter1d, label as ndlabel

SKELETON_LINES = [
    (5,6),(5,11),(6,12),(11,12),
    (5,7),(7,9),(6,8),(8,10),
    (11,13),(13,15),(12,14),(14,16),
    (0,1),(0,2),(1,3),(2,4)
]
SKELETON_CONNECTIONS_3 = [
    (0,1),(1,2),(2,3),(3,4),(1,5),(5,6),(6,7),
    (1,8),(8,9),(9,10),(8,11),(11,12),(12,13),
    (0,14),(0,15),(14,16),(15,16)
]
NUM_KP = 17
IMGSZ  = 640
CONF   = 0.1
IOU    = 0.5

# ── Stage-1 ────────────────────────────────────────────────────────
def get_global_active_window(saliency_values, energy_threshold=0.15):
    threshold  = np.mean(saliency_values) + 0.5 * np.std(saliency_values)
    is_active  = saliency_values > threshold
    labels, nf = ndlabel(is_active)
    if nf == 0:
        return 0, len(saliency_values)-1, 'Static/Uniform'
    total_energy   = np.sum(saliency_values)
    sig_indices    = []
    for i in range(1, nf+1):
        m = (labels == i)
        if np.sum(saliency_values[m]) / total_energy > energy_threshold:
            sig_indices.extend(np.where(m)[0])
    if not sig_indices:
        pk = np.argmax(saliency_values)
        return max(0,pk-2), min(len(saliency_values)-1,pk+2), 'Peak Only'
    return int(min(sig_indices)), int(max(sig_indices)), 'Global Event Zone'


def run_stage1(folder_name, image_paths, stage1_out_dir, vis_dir):
    """
    image_paths : sorted list of 32 image file paths
    Returns stage1_data dict and saves JSON + visualisation.
    """
    frames = []
    for p in image_paths:
        img = cv2.imread(p)
        img = cv2.resize(img, (FRAME_SIZE, FRAME_SIZE))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        frames.append(img)

    frames_np = np.array(frames, dtype=np.float32) / 255.0
    clip = torch.from_numpy(frames_np).permute(3,0,1,2).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits, saliency = stage1_model(clip, return_saliency=True)

    saliency = saliency.squeeze(0).cpu().numpy()
    saliency_interp = np.interp(
        np.linspace(0, len(saliency)-1, CLIP_LEN),
        np.arange(len(saliency)), saliency
    )
    saliency_smooth = gaussian_filter1d(saliency_interp, sigma=1.2)

    start, end, status = get_global_active_window(saliency_smooth)
    varying_k = end - start + 1

    # clip_indices maps position 0-31 → image index (0-based, i.e. same as list index)
    clip_indices = list(range(CLIP_LEN))

    stage1_data = {
        'folder_name':     folder_name,
        'active_window':   [int(start), int(end)],
        'varying_k':       int(varying_k),
        'detection_status': status,
        'saliency_weights': saliency_smooth.tolist(),
        'clip_indices':    clip_indices
    }

    json_path = os.path.join(stage1_out_dir, f'{folder_name}.json')
    with open(json_path, 'w') as fh:
        json.dump(stage1_data, fh, indent=4)

    # ── Visualisation ──
    os.makedirs(vis_dir, exist_ok=True)
    fig = plt.figure(figsize=(18, 10))
    ax = plt.subplot(5, 1, 1)
    ax.plot(saliency_interp, label='Raw', alpha=0.6)
    ax.plot(saliency_smooth, lw=2, label='Smoothed')
    ax.axvspan(start, end, alpha=0.2, color='red', label='Active Window')
    ax.set_title(f'Stage-1 Saliency  |  {folder_name}  |  Window [{start},{end}]')
    ax.set_xlabel('Frame Index'); ax.set_ylabel('Saliency')
    ax.legend(); ax.grid(True)
    cols = 8
    for i in range(CLIP_LEN):
        ax2 = plt.subplot(5, cols, cols + i + 1)
        ax2.imshow(frames[i])
        ax2.set_title(f'{i}\n{saliency_smooth[i]:.3f}', fontsize=7)
        ax2.axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(vis_dir, f'{folder_name}.png'), dpi=100)
    plt.close()

    return stage1_data, image_paths  # image_paths used downstream


# ── Stage-2 ────────────────────────────────────────────────────────
def norm_kp(kp, W, H):
    if kp is None: return [[0.0, 0.0]] * NUM_KP
    return [[float(x/W), float(y/H)] for x,y in kp]

def interpolate_kp(frame_list, dets, num_kp=17):
    L  = len(frame_list)
    xs = np.full((L, num_kp), np.nan)
    ys = np.full((L, num_kp), np.nan)
    for i, f in enumerate(frame_list):
        for j, pt in enumerate(dets[f]):
            if pt[0] is not None: xs[i,j], ys[i,j] = pt[0], pt[1]
    idx = np.arange(L)
    for j in range(num_kp):
        m = ~np.isnan(xs[:,j])
        if m.any():
            xs[:,j] = np.interp(idx, idx[m], xs[m,j])
            ys[:,j] = np.interp(idx, idx[m], ys[m,j])
    return {f: [[float(xs[i,j]), float(ys[i,j])] for j in range(num_kp)]
            for i,f in enumerate(frame_list)}


def run_stage2(folder_name, stage1_data, image_paths, stage2_out_dir, vis_dir):
    """
    Runs YOLOv8 pose on the active-window frames (images, not video).
    """
    active_window = stage1_data['active_window']
    clip_start, clip_end = active_window[0], active_window[1]
    # frame_numbers here are indices into image_paths list
    frame_numbers = list(range(clip_start, clip_end+1))

    # Load frames for those indices
    raw_frames = {}
    frame_sizes = {}
    for fn in frame_numbers:
        img = cv2.imread(image_paths[fn])
        raw_frames[fn]  = img
        frame_sizes[fn] = (img.shape[1], img.shape[0])  # (W, H)

    # Reference size (use first frame)
    ref_img = raw_frames[frame_numbers[0]]
    frame_w, frame_h = ref_img.shape[1], ref_img.shape[0]

    detections_px = {}
    bboxes_px     = {}
    scores        = {}

    os.makedirs(vis_dir, exist_ok=True)

    for fnum in frame_numbers:
        frame = raw_frames[fnum]
        results = yolo_model.predict(source=frame, imgsz=IMGSZ, conf=CONF, iou=IOU, verbose=False)
        vis = frame.copy()
        fw, fh = frame.shape[1], frame.shape[0]

        if len(results) > 0 and len(results[0].boxes) > 0:
            best_idx  = results[0].boxes.conf.argmax()
            kp_data   = results[0].keypoints.data[best_idx].cpu().numpy()
            bbox      = results[0].boxes.xyxy[best_idx].cpu().numpy()
            detections_px[fnum] = kp_data[:, :2]
            scores[fnum]        = float(results[0].boxes.conf[best_idx])
            bboxes_px[fnum]     = [float((bbox[0]+bbox[2])/2), float((bbox[1]+bbox[3])/2),
                                   float(bbox[2]-bbox[0]),     float(bbox[3]-bbox[1])]
            # Draw skeleton
            for s_j, e_j in SKELETON_LINES:
                pt1, pt2 = kp_data[s_j], kp_data[e_j]
                if pt1[2] > 0.3 and pt2[2] > 0.3:
                    cv2.line(vis,(int(pt1[0]),int(pt1[1])),(int(pt2[0]),int(pt2[1])),(255,255,0),2)
            for ki,(x,y,c) in enumerate(kp_data):
                if c > 0.3:
                    color = (0,255,0) if ki > 4 else (255,0,255)
                    cv2.circle(vis,(int(x),int(y)),4,color,-1)
            cv2.rectangle(vis,(int(bbox[0]),int(bbox[1])),(int(bbox[2]),int(bbox[3])),(0,0,255),2)
            cv2.putText(vis,f'Person {scores[fnum]:.2f}',(int(bbox[0]),int(bbox[1])-10),
                        cv2.FONT_HERSHEY_SIMPLEX,0.5,(0,0,255),1)
        else:
            detections_px[fnum] = None
            bboxes_px[fnum]     = None
            scores[fnum]        = 0.0

        cv2.imwrite(os.path.join(vis_dir, f'frame_{fnum:03d}.png'), vis)

    # Normalize
    detections_norm = {f: norm_kp(detections_px[f], frame_w, frame_h) for f in frame_numbers}
    bboxes_norm = {}
    for f in frame_numbers:
        bx = bboxes_px[f]
        bboxes_norm[f] = [bx[0]/frame_w, bx[1]/frame_h, bx[2]/frame_w, bx[3]/frame_h] if bx else [None]*4

    interp_results = interpolate_kp(frame_numbers, detections_norm)

    # Physics features
    frame_data = []
    prev_hip   = None
    for f in frame_numbers:
        kp   = interp_results[f]
        m_sh = [(kp[5][0]+kp[6][0])/2, (kp[5][1]+kp[6][1])/2]
        m_hp = [(kp[11][0]+kp[12][0])/2, (kp[11][1]+kp[12][1])/2]
        tilt = abs(math.degrees(math.atan2(m_sh[0]-m_hp[0], -(m_sh[1]-m_hp[1]))))
        vel  = float(m_hp[1]-prev_hip[1]) if prev_hip else 0.0
        prev_hip = m_hp
        bx   = bboxes_norm[f]
        hw   = float(bx[3]/bx[2]) if bx[2] and bx[2] > 0 else 0.0
        gp   = float(1.0 - m_sh[1])
        frame_data.append({
            'frame_idx':      f,
            'keypoints':      kp,
            'normalized_bbox': bx,
            'score':          scores[f],
            'features': {'tilt_angle': tilt, 'vertical_velocity': vel,
                         'h_w_ratio': hw,    'ground_proximity': gp}
        })

    stage2_data = {'frame_data': frame_data}
    json_path   = os.path.join(stage2_out_dir, f'{folder_name}.json')
    with open(json_path, 'w') as fh:
        json.dump(stage2_data, fh, indent=2)

    # ── Visualisation: montage of skeleton frames ──
    skel_imgs = []
    for fnum in frame_numbers:
        p = os.path.join(vis_dir, f'frame_{fnum:03d}.png')
        if os.path.exists(p):
            skel_imgs.append(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    if skel_imgs:
        cols = min(4, len(skel_imgs))
        rows = math.ceil(len(skel_imgs)/cols)
        fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
        axes = np.array(axes).flatten()
        for ai, simg in enumerate(skel_imgs):
            axes[ai].imshow(simg)
            axes[ai].set_title(f'Frame {frame_numbers[ai]}')
            axes[ai].axis('off')
        for ax in axes[len(skel_imgs):]:
            ax.axis('off')
        plt.suptitle(f'Stage-2 Skeleton  |  {folder_name}')
        plt.tight_layout()
        # Save montage
        montage_dir = os.path.dirname(vis_dir)  # parent of per-folder vis dir
        plt.savefig(os.path.join(montage_dir, f'{folder_name}.png'), dpi=100)
        plt.close()

    return stage2_data


# ── Stage-3 ────────────────────────────────────────────────────────
def load_json_sequence(stage2_data):
    frames = stage2_data.get('frame_data', [])
    sequence, frame_indices = [], []
    for fdata in frames:
        frame_indices.append(fdata.get('frame_idx', -1))
        kp    = np.array(fdata['keypoints'], dtype=np.float32).flatten()
        feat  = fdata.get('features', {})
        f_vec = [
            float(feat.get('tilt_angle',0) or 0) / 180.0,
            float(feat.get('vertical_velocity',0) or 0) / 100.0,
            float(feat.get('h_w_ratio',0) or 0) / 5.0,
            float(feat.get('ground_proximity',0) or 0)
        ]
        combined = np.nan_to_num(np.concatenate([kp, f_vec]), nan=0.0)
        sequence.append(combined)

    real_len    = len(sequence)
    pad_mask    = torch.zeros(SEQ_LEN, dtype=torch.bool)
    sequence_np = np.array(sequence, dtype=np.float32)
    if real_len < SEQ_LEN:
        pad = np.zeros((SEQ_LEN - real_len, NUM_FEATURES))
        sequence_np = np.vstack([sequence_np, pad])
        pad_mask[real_len:] = True
    else:
        sequence_np = sequence_np[:SEQ_LEN]
        real_len = SEQ_LEN

    return (
        torch.FloatTensor(sequence_np).unsqueeze(0).to(DEVICE),
        pad_mask.unsqueeze(0).to(DEVICE),
        frames, frame_indices, real_len
    )


def run_stage3(folder_name, stage2_data, image_paths, stage1_data,
               stage3_out_dir, vis_dir):
    x, mask, frame_entries, frame_indices, real_len = load_json_sequence(stage2_data)

    with torch.no_grad():
        prob, attn_weights = stage3_model(x, mask=mask)

    fall_prob  = float(prob.item())
    risk_level = 'HIGH' if fall_prob >= 0.75 else ('MEDIUM' if fall_prob >= 0.45 else 'LOW')

    aw = attn_weights.cpu().numpy()
    if aw.ndim > 2:
        aw = np.mean(aw, axis=0)
    total_attn = np.sum(aw, axis=0)
    attn_real  = total_attn[:real_len]
    attn_norm  = attn_real / (np.sum(attn_real) + 1e-8)

    stage3_data = {
        'video_name':       folder_name,
        'fall_probability': round(fall_prob, 4),
        'risk_level':       risk_level,
        'keyframe_indices': [int(i) for i in frame_indices[:real_len]],
        'attention_weights': [round(float(w), 4) for w in attn_norm]
    }

    json_path = os.path.join(stage3_out_dir, f'{folder_name}.json')
    with open(json_path, 'w') as fh:
        json.dump(stage3_data, fh, indent=4)

    # ── Visualisation: skeleton overlay on images ──
    os.makedirs(vis_dir, exist_ok=True)
    cols = min(4, real_len)
    rows = math.ceil(real_len / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    axes = np.array(axes).flatten()
    for idx in range(real_len):
        f_num = frame_indices[idx]
        img   = cv2.cvtColor(cv2.imread(image_paths[f_num]), cv2.COLOR_BGR2RGB)
        h, w  = img.shape[:2]
        kps   = np.array(frame_entries[idx]['keypoints'])
        for (a,b) in SKELETON_CONNECTIONS_3:
            if a < len(kps) and b < len(kps):
                pt1 = (int(kps[a][0]*w), int(kps[a][1]*h))
                pt2 = (int(kps[b][0]*w), int(kps[b][1]*h))
                cv2.line(img, pt1, pt2, (0,255,255), 2)
        axes[idx].imshow(img)
        axes[idx].set_title(f'F{f_num}\nW:{attn_norm[idx]:.4f}', fontsize=8)
        axes[idx].axis('off')
    for ax in axes[real_len:]:
        ax.axis('off')
    plt.suptitle(f'Stage-3  |  {folder_name}  |  P(fall)={fall_prob:.4f}  [{risk_level}]')
    plt.tight_layout()
    plt.savefig(os.path.join(vis_dir, f'{folder_name}.png'), dpi=100)
    plt.close()

    return stage3_data


# ── Stage-4 ────────────────────────────────────────────────────────
def get_torso_centroid(keypoints, H, W):
    torso_ids = [5,6,11,12]
    xs, ys = [], []
    for tid in torso_ids:
        xs.append(keypoints[tid][0] * W)
        ys.append(keypoints[tid][1] * H)
    return int(np.median(xs)), int(np.median(ys))

def weighted_avg(values, weights):
    v = np.array(values); w = np.array(weights)
    return np.sum(v*w) / (np.sum(w) + 1e-6)


def run_stage4(folder_name, stage2_data, stage3_data, image_paths,
               stage4_out_dir, vis_dir):
    """
    Generates depth maps from images (not video) using MiDaS,
    then computes temporal physics summary.
    """
    keyframe_indices = stage3_data['keyframe_indices']
    attention_weights = np.array(stage3_data['attention_weights'])
    attention_weights = attention_weights / (np.sum(attention_weights) + 1e-6)

    frame_lookup = {fd['frame_idx']: fd for fd in stage2_data['frame_data']}

    os.makedirs(vis_dir, exist_ok=True)

    tilt_list  = []
    hwr_list   = []
    vel_list   = []
    gp_list    = []
    depth_list = []
    depth_vis_imgs = []

    for i, frame_idx in enumerate(keyframe_indices):
        if frame_idx not in frame_lookup:
            continue
        fd       = frame_lookup[frame_idx]
        features = fd['features']
        tilt_list.append(features['tilt_angle'])
        hwr_list.append(features['h_w_ratio'])
        vel_list.append(features['vertical_velocity'])
        gp_list.append(features['ground_proximity'])

        # Depth from image
        img_path = image_paths[frame_idx]
        img_bgr  = cv2.imread(img_path)
        img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        input_b  = midas_transform(img_rgb).to(DEVICE)
        with torch.no_grad():
            pred = midas(input_b)
            pred = torch.nn.functional.interpolate(
                pred.unsqueeze(1), size=img_rgb.shape[:2],
                mode='bicubic', align_corners=False
            ).squeeze()
        depth = pred.cpu().numpy()
        d_min, d_max = depth.min(), depth.max()
        depth_norm = (depth - d_min) / (d_max - d_min + 1e-6)
        h, w   = depth_norm.shape
        cx, cy = get_torso_centroid(fd['keypoints'], h, w)
        depth_val = float(depth_norm[cy, cx]) if (0 <= cx < w and 0 <= cy < h) else 0.0
        depth_list.append(depth_val)

        # Save individual depth map for later montage
        depth_uint8 = (depth_norm * 255).astype(np.uint8)
        depth_color = cv2.applyColorMap(depth_uint8, cv2.COLORMAP_MAGMA)
        cv2.circle(depth_color, (cx,cy), 6, (0,255,0), -1)
        depth_vis_imgs.append(cv2.cvtColor(depth_color, cv2.COLOR_BGR2RGB))

    if not tilt_list:
        tilt_list = [0.0]; hwr_list = [1.0]; vel_list = [0.0]; gp_list = [0.5]; depth_list = [0.0]

    max_tilt     = float(np.max(tilt_list))
    min_hwr      = float(np.min(hwr_list))
    delta_tilt   = float(tilt_list[-1] - tilt_list[0])
    delta_hwr    = float(hwr_list[-1] - hwr_list[0])
    w_attn       = attention_weights[:len(tilt_list)]
    w_attn       = w_attn / (w_attn.sum() + 1e-6)
    weighted_tilt = float(weighted_avg(tilt_list, w_attn))
    weighted_vel  = float(weighted_avg(vel_list,  w_attn))
    weighted_gp   = float(weighted_avg(gp_list,   w_attn))
    depth_array   = np.array(depth_list)
    depth_variance = float(np.var(depth_array))
    depth_range    = float(np.max(depth_array) - np.min(depth_array))
    depth_drop     = float(np.mean(depth_array[:-1]) - depth_array[-1]) if len(depth_array) > 1 else 0.0

    stage4_data = {
        'video_name':           folder_name,
        'max_tilt':             max_tilt,
        'delta_tilt':           delta_tilt,
        'min_h_w_ratio':        min_hwr,
        'delta_h_w_ratio':      delta_hwr,
        'weighted_tilt':        weighted_tilt,
        'weighted_velocity':    weighted_vel,
        'weighted_ground_proximity': weighted_gp,
        'depth_drop':           depth_drop,
        'depth_variance':       depth_variance,
        'depth_range':          depth_range
    }

    json_path = os.path.join(stage4_out_dir, f'{folder_name}.json')
    with open(json_path, 'w') as fh:
        json.dump(stage4_data, fh, indent=4)

    # ── Visualisation: depth maps montage ──
    if depth_vis_imgs:
        cols = min(4, len(depth_vis_imgs))
        rows = math.ceil(len(depth_vis_imgs) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
        axes = np.array(axes).flatten()
        for ai, dimg in enumerate(depth_vis_imgs):
            axes[ai].imshow(dimg)
            axes[ai].set_title(f'F{keyframe_indices[ai]}\nd={depth_list[ai]:.3f}', fontsize=8)
            axes[ai].axis('off')
        for ax in axes[len(depth_vis_imgs):]:
            ax.axis('off')
        plt.suptitle(f'Stage-4 Depth Maps  |  {folder_name}')
        plt.tight_layout()
        plt.savefig(os.path.join(vis_dir, f'{folder_name}.png'), dpi=100)
        plt.close()

    return stage4_data


# ── Stage-5 ────────────────────────────────────────────────────────
import joblib
import numpy as np

# Load tuned assets
_SVM_MODEL = joblib.load('stage5_svm_model.pkl')
_SVM_SCALER = joblib.load('stage5_svm_scaler.pkl')

def run_stage5_svm_optimized(stage4_data):
    # Same engineering as training
    tilt_vel = stage4_data['max_tilt'] * abs(stage4_data['weighted_velocity'])
    low_hwr_ground = (1 / (stage4_data['min_h_w_ratio'] + 0.1)) * (1 - stage4_data['weighted_ground_proximity'])
    depth_vel = stage4_data['depth_drop'] * stage4_data['weighted_velocity']
    tilt_hwr = stage4_data['max_tilt'] / (stage4_data['min_h_w_ratio'] + 0.1)

    # Construct vector
    raw_features = [
        stage4_data['max_tilt'], stage4_data['delta_tilt'],
        stage4_data['min_h_w_ratio'], stage4_data['delta_h_w_ratio'],
        stage4_data['weighted_tilt'], stage4_data['weighted_velocity'],
        stage4_data['weighted_ground_proximity'], stage4_data['depth_drop'],
        stage4_data['depth_variance'], stage4_data['depth_range'],
        tilt_vel, low_hwr_ground, depth_vel, tilt_hwr
    ]

    # 1. Scale using the PowerTransformer
    vector = np.array([raw_features])
    scaled_vector = _SVM_SCALER.transform(vector)

    # 2. Predict
    prediction = _SVM_MODEL.predict(scaled_vector)[0]
    probs = _SVM_MODEL.predict_proba(scaled_vector)[0]

    return ('FALL' if prediction == 1 else 'NO_FALL'), float(probs[1])


print('✅ All pipeline stage functions defined.')


✅ All pipeline stage functions defined.


In [12]:
# ================================================================
# MAIN PIPELINE — Iterates over all 70 folders in UR_FALL_32
# ================================================================
import csv
import traceback

DATA_ROOT   = '/content/UR_FALL_32'
STAGE1_DIR  = '/content/stage1_output'
STAGE2_DIR  = '/content/stage2_output'
STAGE3_DIR  = '/content/stage3_output'
STAGE4_DIR  = '/content/stage4_output'
CSV_PATH    = '/content/results.csv'

VIS_ROOT    = '/content/visualisation'
VIS_S1      = os.path.join(VIS_ROOT, 'stage1')
VIS_S2      = os.path.join(VIS_ROOT, 'stage2')
VIS_S3      = os.path.join(VIS_ROOT, 'stage3')
VIS_S4      = os.path.join(VIS_ROOT, 'stage4')

for d in [STAGE1_DIR, STAGE2_DIR, STAGE3_DIR, STAGE4_DIR,
          VIS_S1, VIS_S2, VIS_S3, VIS_S4]:
    os.makedirs(d, exist_ok=True)

CSV_HEADER = [
    'folder_name', 'actual_label',
    'Rule1_Strong_Rotation', 'Rule2_Rapid_Tilt_Change',
    'Rule3_Horizontal_Body', 'Rule4_Structural_Collapse',
    'output_label'
]

folders = sorted([
    f for f in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, f))
])

print(f'Found {len(folders)} folders to process.\n')

results_rows = []
errors       = []

for idx, folder_name in enumerate(folders, 1):
    print(f'[{idx:03d}/{len(folders)}] Processing: {folder_name}')

    # ── Label ──
    actual_label = 'FALL' if folder_name.startswith('fall') else 'NO_FALL'

    # ── Collect 32 image paths (already sampled) ──
    folder_path = os.path.join(DATA_ROOT, folder_name)
    image_files = sorted([
        f for f in os.listdir(folder_path)
        if f.lower().endswith('.png') or f.lower().endswith('.jpg')
    ])
    if len(image_files) < 2:
        print(f'  ⚠️  Skipping {folder_name}: only {len(image_files)} images')
        errors.append((folder_name, 'too few images'))
        continue

    image_paths = [os.path.join(folder_path, f) for f in image_files]

    try:
        # ── Stage 1 ──
        stage1_data, _ = run_stage1(
            folder_name, image_paths,
            STAGE1_DIR, VIS_S1
        )

        # ── Stage 2 ──
        # Stage-2 vis: per-folder temp dir for individual skeleton frames
        s2_vis_tmp = os.path.join(VIS_S2, f'_tmp_{folder_name}')
        stage2_data = run_stage2(
            folder_name, stage1_data, image_paths,
            STAGE2_DIR,
            s2_vis_tmp   # individual skeleton frames go here
        )
        # Montage is saved to VIS_S2/<folder_name>.png inside run_stage2

        # ── Stage 3 ──
        stage3_data = run_stage3(
            folder_name, stage2_data, image_paths, stage1_data,
            STAGE3_DIR, VIS_S3
        )

        # ── Stage 4 ──
        stage4_data = run_stage4(
            folder_name, stage2_data, stage3_data, image_paths,
            STAGE4_DIR, VIS_S4
        )

        # ── Stage 5 ──
        decision, fall_confidence = run_stage5(stage4_data)

        row = [
            folder_name, actual_label,
            decision, fall_confidence
        ]
        results_rows.append(row)

        status_str = '✅ CORRECT' if decision == actual_label else '❌ WRONG'
        print(f'  Actual={actual_label}  Predicted={decision}  '
              f'confidence={fall_confidence}  {status_str}\n')

    except Exception as e:
        print(f'  ❌ ERROR: {e}')
        traceback.print_exc()
        errors.append((folder_name, str(e)))

# ── Write CSV ──
with open(CSV_PATH, 'w', newline='') as fh:
    writer = csv.writer(fh)
    writer.writerow(CSV_HEADER)
    writer.writerows(results_rows)

print(f'\n✅ Pipeline complete!')
print(f'   Processed : {len(results_rows)} folders')
print(f'   Errors    : {len(errors)} folders')
print(f'   CSV saved : {CSV_PATH}')
if errors:
    print('   Failed folders:', [e[0] for e in errors])


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ================================================================
# EVALUATION — Confusion Matrix, All Metrics, ROC Curve
# ================================================================
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, roc_auc_score, matthews_corrcoef,
    balanced_accuracy_score, cohen_kappa_score,
    average_precision_score
)
import seaborn as sns

df = pd.read_csv('/content/results.csv')
print(f'Loaded {len(df)} results from CSV\n')
display(df.head(10))

# ── Encode labels ──
# Positive class = FALL (1), Negative class = NO_FALL (0)
y_true  = (df['actual_label']  == 'FALL').astype(int).values
y_pred  = (df['output_label']  == 'FALL').astype(int).values

# Use fall-probability from stage3 for ROC (if available, else use score from rules)
# Fallback: use rule count as a soft score (0-4)
df['rule_score'] = (df['Rule1_Strong_Rotation'] +
                    df['Rule2_Rapid_Tilt_Change'] +
                    df['Rule3_Horizontal_Body'] +
                    df['Rule4_Structural_Collapse'])
y_score = df['rule_score'].values / 4.0   # normalise to [0,1]

# ── Core metrics ──
acc       = accuracy_score(y_true, y_pred)
bal_acc   = balanced_accuracy_score(y_true, y_pred)
prec      = precision_score(y_true, y_pred, zero_division=0)
rec       = recall_score(y_true, y_pred, zero_division=0)
f1        = f1_score(y_true, y_pred, zero_division=0)
spec      = recall_score(y_true, y_pred, pos_label=0, zero_division=0)  # specificity
mcc       = matthews_corrcoef(y_true, y_pred)
kappa     = cohen_kappa_score(y_true, y_pred)
auc_roc   = roc_auc_score(y_true, y_score)
auc_pr    = average_precision_score(y_true, y_score)
cm        = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
ppv = tp / (tp+fp+1e-9)   # precision
npv = tn / (tn+fn+1e-9)   # negative predictive value
fpr = fp / (fp+tn+1e-9)   # false positive rate
fnr = fn / (fn+tp+1e-9)   # false negative rate (miss rate)

metrics_dict = {
    'Accuracy':                  round(acc,   4),
    'Balanced Accuracy':         round(bal_acc, 4),
    'Precision (PPV)':           round(prec,  4),
    'Recall (Sensitivity/TPR)':  round(rec,   4),
    'Specificity (TNR)':         round(spec,  4),
    'F1-Score':                  round(f1,    4),
    'Matthews Corr Coeff (MCC)': round(mcc,   4),
    "Cohen's Kappa":             round(kappa, 4),
    'AUC-ROC':                   round(auc_roc, 4),
    'AUC-PR (Avg Precision)':    round(auc_pr,  4),
    'True Positives (TP)':       int(tp),
    'True Negatives (TN)':       int(tn),
    'False Positives (FP)':      int(fp),
    'False Negatives (FN)':      int(fn),
    'Negative Predictive Value': round(npv, 4),
    'False Positive Rate (FPR)': round(fpr, 4),
    'False Negative Rate (FNR)': round(fnr, 4),
}

print('\n' + '='*55)
print('          CLASSIFICATION METRICS SUMMARY')
print('='*55)
for k, v in metrics_dict.items():
    print(f'  {k:<35} {v}')
print('='*55)

print('\nFull Classification Report (sklearn):')
print(classification_report(y_true, y_pred, target_names=['NO_FALL','FALL']))

# ── Save metrics to CSV ──
metrics_df = pd.DataFrame(list(metrics_dict.items()), columns=['Metric','Value'])
metrics_df.to_csv('/content/metrics_summary.csv', index=False)
print('📊 Metrics saved to /content/metrics_summary.csv')

# ================================================================
# FIGURE 1: Confusion Matrix
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred NO_FALL','Pred FALL'],
            yticklabels=['True NO_FALL','True FALL'],
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Actual Label'); axes[0].set_xlabel('Predicted Label')

# Normalised
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=['Pred NO_FALL','Pred FALL'],
            yticklabels=['True NO_FALL','True FALL'],
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Confusion Matrix (Normalised)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Actual Label'); axes[1].set_xlabel('Predicted Label')

plt.suptitle('Fall Detection — Confusion Matrix', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Saved: /content/confusion_matrix.png')

# ================================================================
# FIGURE 2: ROC Curve
# ================================================================
fpr_arr, tpr_arr, thresholds = roc_curve(y_true, y_score)

fig, ax = plt.subplots(figsize=(7,6))
ax.plot(fpr_arr, tpr_arr, color='darkorange', lw=2,
        label=f'ROC Curve (AUC = {auc_roc:.4f})')
ax.plot([0,1],[0,1], color='navy', lw=1.5, linestyle='--', label='Random Classifier')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (TPR / Recall)', fontsize=12)
ax.set_title('ROC Curve — Fall Detection', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/roc_curve.png', dpi=150)
plt.show()
print('💾 Saved: /content/roc_curve.png')

# ================================================================
# FIGURE 3: Precision-Recall Curve
# ================================================================
from sklearn.metrics import precision_recall_curve
prec_arr, rec_arr, _ = precision_recall_curve(y_true, y_score)

fig, ax = plt.subplots(figsize=(7,6))
ax.plot(rec_arr, prec_arr, color='steelblue', lw=2,
        label=f'PR Curve (AUC = {auc_pr:.4f})')
baseline = y_true.sum() / len(y_true)
ax.axhline(baseline, linestyle='--', color='gray', lw=1.5, label=f'Baseline (prevalence={baseline:.2f})')
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve — Fall Detection', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/pr_curve.png', dpi=150)
plt.show()
print('💾 Saved: /content/pr_curve.png')

# ================================================================
# FIGURE 4: Per-Rule Activation Bar Chart
# ================================================================
rule_cols = ['Rule1_Strong_Rotation','Rule2_Rapid_Tilt_Change',
             'Rule3_Horizontal_Body','Rule4_Structural_Collapse']
rule_labels = ['R1: Strong\nRotation','R2: Rapid\nTilt Change',
               'R3: Horizontal\nBody','R4: Structural\nCollapse']

fall_df   = df[df['actual_label']=='FALL']
nofall_df = df[df['actual_label']=='NO_FALL']

fig, ax = plt.subplots(figsize=(9,5))
x = np.arange(len(rule_cols))
w = 0.35
ax.bar(x - w/2, fall_df[rule_cols].mean(),   w, label='FALL',    color='tomato',    alpha=0.85)
ax.bar(x + w/2, nofall_df[rule_cols].mean(), w, label='NO_FALL', color='steelblue', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(rule_labels, fontsize=11)
ax.set_ylabel('Activation Rate (mean)', fontsize=12)
ax.set_title('Per-Rule Activation Rate by Class', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.1); ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/rule_activation.png', dpi=150)
plt.show()
print('💾 Saved: /content/rule_activation.png')

# ================================================================
# FIGURE 5: Rule Score Distribution
# ================================================================
fig, ax = plt.subplots(figsize=(8,5))
for label, color in [('FALL','tomato'),('NO_FALL','steelblue')]:
    subset = df[df['actual_label']==label]['rule_score']
    ax.hist(subset, bins=[-0.5,0.5,1.5,2.5,3.5,4.5], alpha=0.7,
            color=color, label=label, edgecolor='black', lw=0.8)
ax.axvline(2.5, color='black', linestyle='--', lw=2, label='Decision Boundary (≥3 → FALL)')
ax.set_xlabel('Rule Score (0-4)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Distribution of Rule Scores by Class', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/rule_score_distribution.png', dpi=150)
plt.show()
print('💾 Saved: /content/rule_score_distribution.png')

print('\n✅ All metrics and plots generated successfully.')
